# Prepare the training index

Summarize every training case in one compact paragraph, embed the summary, and persist a resumable CSV index. Prompts are imported from `prompts.py`; model names and credentials come from `.env`.

In [ ]:
from pathlib import Path
import json
import os
import re

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

from prompts import (
    SUMMARY_SYSTEM_PROMPT,
    SUMMARY_USER_TEMPLATE,
)

load_dotenv()

LLM_MODEL = os.getenv("OPENAI_LLM_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is missing. Copy .env.example to .env and add your key.")

client = OpenAI()

## Configuration

Set `REBUILD_INDEX` to `True` after changing the summary prompt or either model.

In [ ]:
TRAIN_DIR = Path("data/train/files")
INDEX_PATH = Path("data/train/train_index.csv")
REBUILD_INDEX = False

case_paths = sorted(TRAIN_DIR.glob("*.md"))
if not case_paths:
    raise FileNotFoundError(f"No Markdown cases found in {TRAIN_DIR.resolve()}")

print(f"Found {len(case_paths)} training cases; output: {INDEX_PATH}")

## OpenAI helpers

In [ ]:
def summarize_case(case_text: str) -> str:
    """Return the one-paragraph retrieval summary shared with the search notebook."""
    response = client.responses.create(
        model=LLM_MODEL,
        instructions=SUMMARY_SYSTEM_PROMPT,
        input=SUMMARY_USER_TEMPLATE.format(case_text=case_text),
    )
    return " ".join(response.output_text.strip().splitlines())


def embed_text(text: str) -> list[float]:
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return response.data[0].embedding

## Build (or resume) the index

Each completed row is written immediately. Embeddings are JSON arrays, which makes them portable inside CSV.

In [ ]:
columns = ["id", "summary", "embedding"]
if INDEX_PATH.exists() and not REBUILD_INDEX:
    index_df = pd.read_csv(INDEX_PATH)
    missing_columns = set(columns) - set(index_df.columns)
    if missing_columns:
        raise ValueError(f"Index is missing columns: {sorted(missing_columns)}")
    rows = index_df[columns].to_dict("records")
    completed_ids = set(index_df["id"].astype(str))
    print(f"Resuming with {len(completed_ids)} existing rows")
else:
    rows = []
    completed_ids = set()

INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)

for position, case_path in enumerate(case_paths, start=1):
    case_id = case_path.stem
    if case_id in completed_ids:
        print(f"[{position}/{len(case_paths)}] skip {case_id}")
        continue

    print(f"[{position}/{len(case_paths)}] process {case_id}")
    case_text = case_path.read_text(encoding="utf-8")
    summary = summarize_case(case_text)
    embedding = embed_text(summary)
    rows.append({
        "id": case_id,
        "summary": summary,
        "embedding": json.dumps(embedding),
    })
    pd.DataFrame(rows, columns=columns).sort_values("id").to_csv(INDEX_PATH, index=False)

index_df = pd.read_csv(INDEX_PATH).sort_values("id").reset_index(drop=True)
print(f"Index ready: {len(index_df)} rows")
index_df[["id", "summary"]].head()

## Validate the generated index

In [ ]:
assert list(index_df.columns) == columns
assert index_df["id"].is_unique
assert not index_df[columns].isna().any().any()

vectors = [np.asarray(json.loads(value), dtype=float) for value in index_df["embedding"]]
assert vectors and all(vector.ndim == 1 and vector.size == vectors[0].size for vector in vectors)
print(f"Validated {len(vectors)} embeddings with {vectors[0].size} dimensions")